# ML-Enhanced Pairs Trading Example

This notebook demonstrates the complete workflow for implementing and backtesting an ML-enhanced pairs trading strategy.

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Import custom modules
sys.path.append('..')
from src.data.data_fetcher import DataFetcher
from src.data.pair_selector import PairSelector
from src.models.lstm_model import LSTMSpreadPredictor
from src.models.regime_classifier import RegimeClassifier
from src.strategies.ml_pairs_strategy import MLEnhancedPairsStrategy
from src.backtesting.backtester import PairsBacktester
from src.utils.visualization import (
    plot_cointegration_test,
    plot_pairs_correlation_matrix,
    plot_regime_analysis,
    plot_feature_importance
)

## 1. Configuration

In [ ]:
# Define stock pair
TICKER1 = 'GLD'  # Gold ETF
TICKER2 = 'GDX'  # Gold Miners ETF

# Date range
START_DATE = (datetime.now() - timedelta(days=730)).strftime('%Y-%m-%d')
END_DATE = datetime.now().strftime('%Y-%m-%d')

# Capital
INITIAL_CAPITAL = 100000

print(f"Analyzing pair: {TICKER1} / {TICKER2}")
print(f"Period: {START_DATE} to {END_DATE}")
print(f"Initial Capital: ${INITIAL_CAPITAL:,}")

## 2. Fetch Historical Data

In [ ]:
# Initialize data fetcher
data_fetcher = DataFetcher(start_date=START_DATE, end_date=END_DATE)

# Fetch data
prices = data_fetcher.fetch_stock_data([TICKER1, TICKER2])

print(f"Downloaded {len(prices)} days of data")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")

# Display first few rows
prices.head()

## 3. Test Cointegration

In [ ]:
# Initialize pair selector
pair_selector = PairSelector()

# Get pair metrics
metrics = pair_selector.get_pair_metrics(prices[TICKER1], prices[TICKER2])

print("Cointegration Test Results:")
print(f"  Cointegration p-value: {metrics['cointegration_pvalue']:.4f}")
print(f"  Is cointegrated: {metrics['is_cointegrated']}")
print(f"  Hedge ratio: {metrics['hedge_ratio']:.4f}")
print(f"  Spread ADF p-value: {metrics['spread_adf_pval']:.4f}")
print(f"  Is spread stationary: {metrics['is_spread_stationary']}")

spread = metrics['spread']
zscore = metrics['zscore']

## 4. Visualize Pair Relationship

In [ ]:
# Plot cointegration analysis
plot_cointegration_test(
    prices[TICKER1], 
    prices[TICKER2], 
    spread, 
    zscore,
    TICKER1, 
    TICKER2
)

## 5. Initialize ML-Enhanced Strategy

In [ ]:
# Initialize strategy
strategy = MLEnhancedPairsStrategy(
    entry_zscore=2.0,
    exit_zscore=0.5,
    stop_loss_zscore=3.0,
    use_lstm=True,
    use_regime=True,
    use_rl=False
)

print("Strategy Configuration:")
print(f"  Entry Z-score: ±{strategy.entry_zscore}")
print(f"  Exit Z-score: ±{strategy.exit_zscore}")
print(f"  Stop loss Z-score: ±{strategy.stop_loss_zscore}")
print(f"  LSTM enabled: {strategy.use_lstm}")
print(f"  Regime classifier enabled: {strategy.use_regime}")
print(f"  RL agent enabled: {strategy.use_rl}")

## 6. Run Backtest

In [ ]:
# Initialize backtester
backtester = PairsBacktester(
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=0.001
)

# Run backtest
print("Running backtest (this may take a few minutes)...")
results = backtester.run_backtest(
    strategy=strategy,
    prices1=prices[TICKER1],
    prices2=prices[TICKER2],
    spread=spread,
    zscore=zscore,
    train_split=0.7
)

print(f"Backtest completed - {len(results)} periods simulated")

## 7. Performance Analysis

In [ ]:
# Print summary
backtester.print_summary()

In [ ]:
# Plot results
backtester.plot_results()

## 8. Analyze ML Model Performance

In [ ]:
# Feature importance from regime classifier
if strategy.use_regime:
    feature_importance = strategy.regime_classifier.get_feature_importance()
    
    print("\nTop 10 Most Important Features for Regime Classification:")
    sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)[:10]
    for i, (feature, importance) in enumerate(sorted_features, 1):
        print(f"  {i}. {feature}: {importance:.4f}")
    
    # Plot feature importance
    plot_feature_importance(
        feature_importance,
        title="Regime Classifier Feature Importance"
    )

## 9. Compare with Classical Strategy

In [ ]:
# Run classical strategy (no ML)
classical_strategy = MLEnhancedPairsStrategy(
    entry_zscore=2.0,
    exit_zscore=0.5,
    stop_loss_zscore=3.0,
    use_lstm=False,
    use_regime=False,
    use_rl=False
)

classical_backtester = PairsBacktester(
    initial_capital=INITIAL_CAPITAL,
    transaction_cost=0.001
)

print("Running classical strategy backtest...")
classical_results = classical_backtester.run_backtest(
    strategy=classical_strategy,
    prices1=prices[TICKER1],
    prices2=prices[TICKER2],
    spread=spread,
    zscore=zscore,
    train_split=0.7
)

# Compare metrics
ml_metrics = backtester.calculate_metrics()
classical_metrics = classical_backtester.calculate_metrics()

comparison = pd.DataFrame({
    'Classical': classical_metrics,
    'ML-Enhanced': ml_metrics
})

print("\nStrategy Comparison:")
print(comparison)

## 10. Conclusions

Key takeaways:
1. The ML-enhanced strategy typically shows improved risk-adjusted returns
2. Regime classification helps avoid unfavorable market conditions
3. LSTM predictions can filter out false signals
4. Transaction costs significantly impact performance
5. Careful parameter tuning is essential for optimal results